# ucon v0.10.0: Scientific Computing

This notebook demonstrates ucon's NumPy, Pandas, and Polars integrations for batch operations on dimensioned quantities.

In [1]:
# Install optional dependencies if needed
# !pip install ucon[numpy,pandas,polars]

In [2]:
import numpy as np
from ucon import units, Scale
from ucon.numpy import NumberArray

# Create NumberArray from a list
heights = units.meter([1.65, 1.70, 1.75, 1.80, 1.85, 1.90])
print(f"Heights: {heights}")
print(f"Shape: {heights.shape}")
print(f"Length: {len(heights)}")

Heights: <[1.65, 1.7 , 1.75, 1.8 , 1.85, 1.9 ] m>
Shape: (6,)
Length: 6


## 1. NumberArray Basics

Create arrays of dimensioned quantities using the familiar callable syntax.

In [2]:
# Create from numpy array
km = Scale.kilo * units.meter
data = np.linspace(0, 100, 11)
distances = km(data)
print(f"Distances: {distances}")

Distances: <[0, 10, 20, ..., 80, 90, 100] km>


In [3]:
# Indexing: scalar index returns Number, slice returns NumberArray
print(f"First height: {heights[0]}")
print(f"Type: {type(heights[0]).__name__}")
print()
print(f"First three: {heights[:3]}")
print(f"Type: {type(heights[:3]).__name__}")

First height: <1.65 m>
Type: Number

First three: <[1.65, 1.7 , 1.75] m>
Type: NumberArray


In [4]:
# Iteration yields Number instances
for i, h in enumerate(heights[:3]):
    print(f"  Person {i+1}: {h}")

  Person 1: <1.65 m>
  Person 2: <1.7 m>
  Person 3: <1.75 m>


## 2. Vectorized Conversion

Convert entire arrays between units in a single operation.

In [5]:
# Convert heights to feet
heights_ft = heights.to(units.foot)
print(f"Heights in feet: {heights_ft}")

Heights in feet: <[5.4134, 5.5774, 5.7415, 5.9055, 6.0696, 6.2336] ft>


In [6]:
# Temperature conversion (affine map)
temps_c = units.celsius([0, 20, 37, 100])
temps_f = temps_c.to(units.fahrenheit)
print(f"Celsius:    {temps_c}")
print(f"Fahrenheit: {temps_f}")

Celsius:    <[  0.,  20.,  37., 100.] °C>
Fahrenheit: <[ 32. ,  68. ,  98.6, 212. ] °F>


In [7]:
# Scale conversions
km = Scale.kilo * units.meter
distances_km = km([1, 5, 10, 42.195])  # marathon!
distances_mi = distances_km.to(units.mile)
print(f"Kilometers: {distances_km}")
print(f"Miles:      {distances_mi}")

Kilometers: <[ 1.   ,  5.   , 10.   , 42.195] km>
Miles:      <[ 0.6214,  3.1069,  6.2137, 26.2188] mi>


## 3. Vectorized Arithmetic

Arithmetic operations preserve units and combine dimensions correctly.

In [8]:
# Same-unit addition
a = units.meter([1, 2, 3])
b = units.meter([4, 5, 6])
print(f"a + b = {a + b}")
print(f"b - a = {b - a}")

a + b = <[5., 7., 9.] m>
b - a = <[3., 3., 3.] m>


In [3]:
# Multiplication creates derived units
lengths = units.meter([2, 3, 4])
widths = units.meter([5, 6, 7])
areas = lengths * widths
print(f"Areas: {areas}")

Areas: <[10., 18., 28.] m²>


In [4]:
# Division creates ratios
distances = units.meter([100, 200, 400])
times = units.second([10, 20, 50])
speeds = distances / times
print(f"Speeds: {speeds}")

Speeds: <[10., 10.,  8.] m/s>


In [5]:
# Scalar multiplication
doubled = heights * 2
print(f"Doubled heights: {doubled}")

Doubled heights: <[3.3, 3.4, 3.5, 3.6, 3.7, 3.8] m>


## 4. Reduction Operations

Statistical operations preserve units.

In [6]:
measurements = units.meter([10.2, 10.5, 10.3, 10.4, 10.6, 10.1])

print(f"Sum:  {measurements.sum()}")
print(f"Mean: {measurements.mean()}")
print(f"Std:  {measurements.std()}")
print(f"Min:  {measurements.min()}")
print(f"Max:  {measurements.max()}")

Sum:  <62.1 m>
Mean: <10.35 m>
Std:  <0.1707825127659934 m>
Min:  <10.1 m>
Max:  <10.6 m>


## 5. Uncertainty Propagation

Track measurement uncertainty through calculations.

In [7]:
# Uniform uncertainty (same for all elements)
measurements = units.meter([10.0, 20.0, 30.0, 40.0], uncertainty=0.5)
print(f"Measurements: {measurements}")

Measurements: <[10., 20., 30., 40.] ± 0.5 m>


In [8]:
# Per-element uncertainty
from ucon.numpy import NumberArray

values = np.array([100.0, 200.0, 300.0])
errors = np.array([1.0, 2.0, 1.5])
data = NumberArray(values, unit=units.meter, uncertainty=errors)
print(f"Data with per-element uncertainty: {data}")

Data with per-element uncertainty: <[100., 200., 300.] ± [...] m>


In [9]:
# Uncertainty propagates through arithmetic (quadrature)
a = units.meter([10, 20], uncertainty=0.5)
b = units.meter([5, 10], uncertainty=0.3)

result = a + b
print(f"a + b = {result}")
print(f"Combined uncertainty: {result.uncertainty}")

a + b = <[15., 30.] ± 0.5831 m>
Combined uncertainty: 0.58309518948453


In [10]:
# Uncertainty in reductions
data = units.meter([1, 2, 3, 4], uncertainty=0.1)

total = data.sum()
print(f"Sum: {total}")
print(f"Sum uncertainty: {total.uncertainty:.4f} (0.1 * sqrt(4) = 0.2)")

avg = data.mean()
print(f"Mean: {avg}")
print(f"Mean uncertainty: {avg.uncertainty:.4f} (0.1 / sqrt(4) = 0.05)")

Sum: <10.0 ± 0.2 m>
Sum uncertainty: 0.2000 (0.1 * sqrt(4) = 0.2)
Mean: <2.5 ± 0.05 m>
Mean uncertainty: 0.0500 (0.1 / sqrt(4) = 0.05)


In [11]:
# Uncertainty through conversion
heights = units.meter([1.80, 1.85], uncertainty=0.01)
heights_ft = heights.to(units.foot)
print(f"Heights (m):  {heights}")
print(f"Heights (ft): {heights_ft}")
print(f"Uncertainty scales with conversion factor")

Heights (m):  <[1.8 , 1.85] ± 0.01 m>
Heights (ft): <[5.9055, 6.0696] ± 0.03281 ft>
Uncertainty scales with conversion factor


## 6. Comparison and Filtering

Comparison operators return boolean arrays for filtering.

In [12]:
heights = units.meter([1.65, 1.70, 1.75, 1.80, 1.85, 1.90])

# Compare with scalar
tall = heights > 1.75
print(f"Heights > 1.75m: {tall}")

Heights > 1.75m: [False False False  True  True  True]


In [13]:
# Compare with Number
threshold = units.meter(1.80)
above = heights >= threshold
print(f"Heights >= 1.80m: {above}")

Heights >= 1.80m: [False False False  True  True  True]


In [14]:
# Use for filtering
tall_heights = heights.quantities[heights > 1.75]
print(f"Tall people's heights (raw): {tall_heights}")

Tall people's heights (raw): [1.8  1.85 1.9 ]


## 7. N-Dimensional Arrays

NumberArray supports arrays of any dimension.

In [15]:
# 2D grid of measurements
grid = NumberArray(
    [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]],
    unit=units.meter
)
print(f"Grid shape: {grid.shape}")
print(f"Grid ndim: {grid.ndim}")
print(f"Grid:\n{grid}")

Grid shape: (3, 3)
Grid ndim: 2
Grid:
<[[1., 2., 3.],
 [4., 5., 6.],
 [7., 8., 9.]] m>


In [16]:
# Row indexing
print(f"First row: {grid[0]}")
print(f"Element [1,2]: {grid[1, 2]}")

First row: <[1., 2., 3.] m>
Element [1,2]: <6.0 m>


In [17]:
# Broadcasting
row = NumberArray([10, 20, 30], unit=units.meter)
result = grid + row
print(f"Grid + row (broadcast):\n{result}")

Grid + row (broadcast):
<[[11., 22., 33.],
 [14., 25., 36.],
 [17., 28., 39.]] m>


## 8. Pandas Integration

Work with unit-aware DataFrame columns.

In [18]:
import pandas as pd
from ucon.pandas import NumberSeries

# Create a DataFrame with measurements
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'height_m': [1.65, 1.80, 1.75, 1.70],
    'weight_kg': [60, 80, 75, 65]
})
df

,name,height_m,weight_kg
0,Alice,1.65,60
1,Bob,1.80,80
2,Charlie,1.75,75
3,Diana,1.70,65


In [19]:
# Wrap column with unit using accessor
heights = df['height_m'].ucon.with_unit(units.meter)
print(f"Heights: {heights}")

Heights: <NumberSeries [1.65, 1.8, 1.75, 1.7] m>


In [20]:
# Convert to different unit
heights_ft = df['height_m'].ucon.with_unit(units.meter).to(units.foot)
print(f"Heights in feet: {heights_ft}")

Heights in feet: <NumberSeries [5.413, 5.906, 5.741, 5.577] ft>


In [21]:
# Add converted column to DataFrame
df['height_ft'] = heights_ft.series
df

,name,height_m,weight_kg,height_ft
0,Alice,1.65,60,5.413386
1,Bob,1.80,80,5.905512
2,Charlie,1.75,75,5.741470
3,Diana,1.70,65,5.577428


In [22]:
# NumberSeries arithmetic
kg = Scale.kilo * units.gram
heights = NumberSeries(df['height_m'], unit=units.meter)
weights = NumberSeries(df['weight_kg'], unit=kg)

# Calculate BMI (kg/m^2)
bmi = weights / (heights * heights)
print(f"BMI: {bmi}")

BMI: <NumberSeries [22.04, 24.69, 24.49, 22.49] kg/m²>


## 9. Polars Integration

Unit-aware columns for Polars DataFrames.

In [23]:
import polars as pl
from ucon.polars import NumberColumn

# Create a Polars DataFrame
df_pl = pl.DataFrame({
    'distance_km': [5.0, 10.0, 21.1, 42.2],
    'time_min': [25.0, 55.0, 120.0, 260.0]
})
df_pl

distance_km,time_min
f64,f64
5.0,25.0
10.0,55.0
21.1,120.0
42.2,260.0


In [24]:
# Wrap columns with units
km = Scale.kilo * units.meter
distances = NumberColumn(df_pl['distance_km'], unit=km)
print(f"Distances: {distances}")

Distances: <NumberColumn [5, 10, 21.1, 42.2] km>


In [25]:
# Convert to miles
distances_mi = distances.to(units.mile)
print(f"Distances in miles: {distances_mi}")

Distances in miles: <NumberColumn [3.107, 6.214, 13.11, 26.22] mi>


In [26]:
# Calculate pace (time per distance)
times = NumberColumn(df_pl['time_min'], unit=units.minute)
pace = times / distances
print(f"Pace: {pace}")

Pace: <NumberColumn [5, 5.5, 5.687, 6.161] min/km>


## 10. Real-World Example: Experimental Data Analysis

Combining all features for a realistic workflow.

In [ ]:
# Simulate experimental measurements with uncertainty
np.random.seed(42)

# True value: 9.81 m/s^2
n_measurements = 20
true_g = 9.81
measurement_error = 0.05  # 5 cm/s^2 instrument precision

measured_values = true_g + np.random.normal(0, measurement_error, n_measurements)

# Inject a couple of outliers (simulating measurement errors)
measured_values[5] = 9.55   # low outlier
measured_values[12] = 10.05  # high outlier

# Create NumberArray with uncertainty
g_measurements = NumberArray(
    measured_values,
    unit=units.meter / units.second**2,
    uncertainty=measurement_error
)

print(f"Measurements: {g_measurements}")

In [28]:
# Statistical analysis
mean_g = g_measurements.mean()
std_g = g_measurements.std()

print(f"Mean: {mean_g}")
print(f"Std Dev: {std_g}")
print(f"True value: {true_g} m/s^2")
print(f"Error: {abs(mean_g.quantity - true_g):.4f} m/s^2")

Mean: <9.801435071927909 ± 0.011180339887498949 m/s²>
Std Dev: <0.0467859979446887 m/s²>
True value: 9.81 m/s^2
Error: 0.0086 m/s^2


In [ ]:
# Identify outliers (> 2 sigma from mean)
deviations = np.abs(g_measurements.quantities - mean_g.quantity)
outliers = deviations > 2 * std_g.quantity

print(f"Outlier mask: {outliers}")
print(f"Number of outliers: {np.sum(outliers)}")

In [ ]:
# Filter outliers and recalculate
filtered_values = g_measurements.quantities[~outliers]
g_filtered = NumberArray(
    filtered_values,
    unit=units.meter / units.second**2,
    uncertainty=measurement_error
)

print(f"Filtered mean: {g_filtered.mean()}")
print(f"Filtered std:  {g_filtered.std()}")

In [ ]:
# Convert to different units for reporting
cm = Scale.centi * units.meter
g_cm = g_filtered.to(cm / units.second**2)
print(f"In cm/s^2: {g_cm.mean()}")

## Summary

ucon v0.10.0 provides seamless integration with the scientific Python ecosystem:

- **NumberArray**: Vectorized operations on dimensioned arrays with uncertainty tracking
- **Pandas**: `NumberSeries` and `.ucon` accessor for DataFrame workflows
- **Polars**: `NumberColumn` for high-performance DataFrames
- **Performance**: Lazy caching makes repeated operations fast

Install with optional dependencies:
```bash
pip install ucon[numpy]           # Just NumPy
pip install ucon[pandas]          # Pandas integration
pip install ucon[polars]          # Polars integration
pip install ucon[numpy,pandas,polars]  # All integrations
```